In [ ]:
#remove warnings
import warnings
warnings.filterwarnings('ignore')

#matrix manipulation
import pandas as pd

#scrapping data
import cloudscraper
from bs4 import BeautifulSoup

### Historical Players (1946 - 2024)

In [ ]:
years = list(range(1946, 2024))

In [ ]:
player_stats_url = 'https://www.basketball-reference.com/leagues/NBA_{}_advanced.html'

In [ ]:
scraper = cloudscraper.create_scraper() 

for year in years:
    url = player_stats_url.format(year)
    data = scraper.get(url)
    with open('html/{}.html'.format(year), 'w+',  encoding='utf-8') as f:
        f.write(data.text)

In [ ]:
dfs = []
for year in years:
    with open('html/{}.html'.format(year), encoding="utf-8") as f:
        page = f.read()

    soup = BeautifulSoup(page, "html.parser")
    player_table = soup.find(id="switcher_per_game_stats")
    player = pd.read_html(str(player_table))[0]
    player['Year'] = year

    dfs.append(player)

    players = pd.concat(dfs)

    players = players.dropna(axis=0, subset=['Age'])

    players.to_csv("csv/{}.csv".format(year))

    players.reset_index(inplace=True)

    players.to_json("json/{}.json".format(year), orient='records', lines=True)